## Grid SANOS Market Generation Test

This notebook computes historic time series for implied vols, fitting error, DLV vs a gixed NMS grid.



In [1]:

import os
os.environ["JAX_ENABLE_X64"] = "True"

%load_ext autoreload
%autoreload 1

%aimport quantdata
%aimport quantdata.biz_calendar
%aimport quantdata.widb
%aimport quantdata.widb.client
%aimport quantdata.widb.day
%aimport quantdata.dysanos
%aimport quantdata.utils.backbone
%aimport quantdata.sanos.sanos_tools
%aimport quantdata.sanos.arbfree
%aimport quantdata.sanos.sanos
%aimport quantdata.sanos.sanos_jax


In [2]:

import math

from cdxcore.bs import bs
from cdxcore.dynaplot import figure, colors_tableau
from cdxcore.subdir import SubDir
from cdxcore.uniquehash import unique_hash8
from cdxcore.verbose import Context
from cdxcore.version import version
from matplotlib import cm
from matplotlib.colors import Normalize
from scipy.stats import norm
from quantdata.biz_calendar import DAYS_PER_YEAR, to_date, NMS5
from cdxcore.pretty import PrettyObject
import numpy as np
from dataclasses import dataclass, field    
import warnings
import scipy as scipy
from collections.abc import Callable
import ipynbname
from quantdata.sanos.sanos import SanosFitter
from quantdata.sanos.sanos_jax import fit_dlvsqs

from quantdata.sanos.arbfree import MkArbFree
from datetime import date

DEFAULT_EXPIRIES = np.array([1./255., 3./255., 5./255., 10./255., 20./255., 40./255.,0.5,1.], dtype=np.float64)
DEFAULT_EXPIRIES = np.array([2./255., 5./255., 10./255., 40./255.,0.5,1.], dtype=np.float64)
DEFAULT_NUMS     = (-2,1)

_cache = SubDir("!/.cache_"+ipynbname.name(), fmt=SubDir.PICKLE)

@version("0.1")
@dataclass(frozen=True)
class SanosBenchmarkConfig:
    smoothness     : float = 0.25
    grid_expiries  : np.array = field( default_factory=lambda: DEFAULT_EXPIRIES )
    grid_nms       : tuple = DEFAULT_NUMS
    grid_num_nms   : int = 10
    min_dlv        : float = 0.01    # Minimum DLV used for linear programming. Is *not* imposed on the output DLV to avoid numerical discrepancies between DLV-implied and actual densities.
    max_dlv        : float = 2.      # Maximum DLV used for linear programming. Is *not* imposed on the output DLV to avoid numerical discrepancies between DLV-implied and actual densities.

    def _generate_grid_nms(self):
        """ Generate grid which contains '1' """
        dist = float(self.grid_nms[1] - self.grid_nms[0])
        Nlo  = min(max(2, float(self.grid_num_nms*abs(self.grid_nms[0]))//dist),self.grid_num_nms-3)
        Nhi = self.grid_num_nms - Nlo - 1

        assert Nlo>=2 and Nhi>=2, f"Internal error: error {Nlo} {Nhi} for grid_nms {self.grid_nms} and grid_num_nms {self.grid_num_nms}."
        grid_nms = np.concatenate( ( np.linspace(self.grid_nms[0], 0., int(Nlo), endpoint=False), np.linspace(0., self.grid_nms[1], int(Nhi)+1) ) )
        assert len(grid_nms)==self.grid_num_nms, f"Internal error: generated number of moneyness levels {len(grid_nms)} does not match expected {self.grid_num_nms} for grid_nms {self.grid_nms} and grid_num_nms {self.grid_num_nms}."
        return grid_nms

    @_cache.cache(
        "0.14", dependencies=[MkArbFree, SanosFitter],
        uid=lambda self, ticker, d, market : f"{unique_hash8(self)} {market.uid}/{ticker}/{d}")
    def __call__( self, ticker : str, d : date, market, *, message_context : Callable|str|None = None, verbose = Context.all ):

        def message(with_cols : bool = True) -> str:
            if message_context is None:
                return ""
            if callable(message_context):
                msg = message_context()
            else:
                msg = str(message_context)
            if msg == "":
                return msg
            msg = msg[0].upper() + msg[1:]
            return msg if not with_cols else msg + ": "

        assert set(market) >= {'expiries', 'strikes', 'bids', 'asks'}, f"Expected market to have keys 'expiries', 'strikes', 'bids', 'asks' but found {set(market)}"

        str_smth = f"{self.smoothness:.2f}-smooth" if self.smoothness is not None else "linear"

        # fit a smooth SANOS model to the market
        # --------------------------------------

        with verbose.write_t(lambda : f"\r{message()}Fitting {str_smth} SANOS to {sum( len(s) for s in market.strikes )} strikes... ", end="") as tme:
            mk_arb_free  = MkArbFree(smoothness=self.smoothness, surface_expiry_linear=True, min_dlv=self.min_dlv, max_dlv=self.max_dlv )
            af_output    = mk_arb_free( strikes = market.strikes, expiries = market.expiries, bids = market.bids, asks = market.asks, verbose=verbose.quiet )
            fit_result   = af_output.compute_market_fit( expiries=market.expiries, strikes=market.strikes, bids=market.bids, asks=market.asks, and_vols=False )
            verbose.write(f"done; median error is {fit_result.median_err:.2f} half-spreads for the {fit_result.pcnt_fit*100:.1f}% of options which were not fitted within bid-ask spread. This took {tme}.", head=False )

        # NMS grid
        # --------

        nms_grid = PrettyObject()

        with verbose.write_t(lambda : f"\rTraining {str_smth} SANOS on NMS {self.grid_nms} grid:") as tme:
            if self.grid_expiries[-1] > market.expiries[-1]:
                warnings.warn(f"{message()}Warning {str_smth} SANOS: the maximum grid expiry {self.grid_expiries[-1]:.4g} is greater than the maximum market expiry {market.expiries[-1]:.4g}.")

            M                   = len( self.grid_expiries )
            N                   = self.grid_num_nms
            nms_grid.nms        = self._generate_grid_nms()
            nms_grid.expiries   = np.asarray( self.grid_expiries )
            nms_grid.atm_vols   = af_output.interpolate_atm_vols( nms_grid.expiries )
            nms_grid.fwd_vols   = np.sqrt( np.diff(nms_grid.atm_vols**2 * nms_grid.expiries,prepend=0.) / np.diff(nms_grid.expiries, prepend=0.) )
            assert np.all(np.isfinite(nms_grid.fwd_vols)), f"Internal error: generated non-finite ATM fwd vols {nms_grid.fwd_vols} for grid expiries {nms_grid.expiries} from input expiries {af_output.expiries} and variances {af_output.variances}."

            eps_vol             = af_output.interpolate_atm_vols( NMS5.EPS ) # not sure we need that
            nms_grid.strikes    = NMS5.from_nms( z=nms_grid.nms[None,:], t=nms_grid.expiries[:,None], vol=nms_grid.atm_vols[:,None], eps_vol=eps_vol ) # (M,N)
            nms_grid.mids,\
            nms_grid.mid_vols   = af_output.price( expiries=nms_grid.expiries[:,None], strikes=nms_grid.strikes, and_vols=True, mode="surface" )
            nms_grid.xstrikes   = np.zeros((M,self.grid_num_nms+2), dtype=nms_grid.strikes.dtype)
            nms_grid.xstrikes[:,1:-1] = nms_grid.strikes
            nms_grid.xstrikes[:,0]    = nms_grid.strikes[:,0]/2.
            nms_grid.xstrikes[:,-1]   = nms_grid.strikes[:,-1]*1.5

            # fit and price market4
            sanos           = SanosFitter(smoothness=self.smoothness, min_dlv=self.min_dlv, max_dlv=self.max_dlv )
            sn_grid         = sanos(    expiries = nms_grid.expiries, 
                                        strikes  = [ nms_grid.strikes[t,:] for t in range(M) ],
                                        mids     = [ nms_grid.mids[t,:] for t in range(M) ], 
                                        xstrikes = [ nms_grid.xstrikes[t,:] for t in range(M) ],
                                        vols     = nms_grid.atm_vols,
                                        message_context = lambda : message(with_cols=False),
                                        verbose=verbose(1)) 
            sn_grid         = fit_dlvsqs( sanos,
                                        expiries = nms_grid.expiries, 
                                        strikes  = [ nms_grid.strikes[t,:] for t in range(M) ],
                                        mids     = [ nms_grid.mids[t,:] for t in range(M) ], 
                                        xstrikes = [ nms_grid.xstrikes[t,:] for t in range(M) ],
                                        #vols     = nms_grid.atm_vols,
                                        #debug_do_not_optimize = False,
                                        init_dlvs = [ np.sqrt(_) for _ in sn_grid.dlvsqs() ],
                                        message_context = lambda : message(with_cols=False),
                                        verbose=verbose(1)) 

            nms_grid.fits,\
            nms_grid.fit_vols    = sn_grid( expiries=nms_grid.expiries[:,None], strikes=nms_grid.strikes, and_vols=True )

            err_vols             = np.median( np.abs( nms_grid.fit_vols - nms_grid.mid_vols ) )
            nms_grid.fit_market  = sn_grid.compute_market_fit( expiries=market.expiries, strikes=market.strikes, bids=market.bids, asks=market.asks, and_vols = False )
            nms_grid.dlvsqs      = sn_grid.dlvsqs( min_dlv=0., max_dlv=None, on_all_strikes=False ) # do not use DLV bounds here!
            verbose.write(f"done; median error vs market is {nms_grid.fit_market.median_err:.2f} half-spreads for the {nms_grid.fit_market.pcnt_fit*100:.1f}% of options which were not within bid-ask spread. "+\
                          f"Note that the model was fitted only vs the NMS grid, not the market. This median match to implied vols was {err_vols:.2f}. This took {tme}.")

            assert isinstance( nms_grid.fit_vols, np.ndarray ) , f"Internal error: expected nms_grid.fit_vols to be a numpy array but found {type(nms_grid.fit_vols)}."
            assert isinstance( nms_grid.dlvsqs, list), f"Internal error: expected nms_grid.dlvsqs to be a list but found {type(nms_grid.dlvsqs)}."
            nms_grid.dlvsqs       = np.stack( nms_grid.dlvsqs, axis=0 )
            assert nms_grid.fit_vols.shape == (M,N), f"Internal error: expected nms_grid.fit_vols to have shape {(M,N)} but found {nms_grid.fit_vols.shape}."
            assert nms_grid.dlvsqs.shape == (M,N), f"Internal error: expected nms_grid.dlvsqs to have shape {(M,N)} but found {nms_grid.dlvsqs.shape}."
            assert nms_grid.strikes.shape == (M,N), f"Internal error: expected nms_grid.strikes to have shape {(M,N)} but found {nms_grid.strikes.shape}."
            assert nms_grid.atm_vols.shape == (M,), f"Internal error: expected nms_grid.atm_vols to have shape {(M,)} but found {nms_grid.atm_vols.shape}."
            assert nms_grid.fwd_vols.shape == (M,), f"Internal error: expected nms_grid.fwd_vols to have shape {(M,)} but found {nms_grid.fwd_vols.shape}."


        # error of grid SANOS vs market
        r = PrettyObject()
        r.mktfit_err_pcnt   = np.array( [fit_result.pcnt_fit] )              # (1,)
        r.mktfit_median_err = np.array( [fit_result.median_err] )            # (1,)
        r.mktfit_mean_err   = np.array( [fit_result.mean_err] )              # (1,)
        r.mktfit_fits       = fit_result.fits
        r.grid_err_pcnt     = np.array( [nms_grid.fit_market.pcnt_fit] )     # (1,)
        r.grid_median_err   = np.array( [nms_grid.fit_market.median_err] )   # (1,)
        r.grid_mean_err     = np.array( [nms_grid.fit_market.mean_err] )     # (1,)
        r.atm_vols     = nms_grid.atm_vols                              # (M,)
        r.fwd_vols     = nms_grid.fwd_vols                              # (M,)
        r.expiries     = nms_grid.expiries                              # (M,)
        r.nms          = nms_grid.nms                                   # (N,)
        r.fit_vols     = nms_grid.fit_vols                              # (M,N)
        r.mid_vols     = nms_grid.mid_vols                              # (M,N)
        r.dlvsqs       = nms_grid.dlvsqs                                # (M,N) 
        r.strikes      = nms_grid.strikes                              # (M,N)
        r.pricer       = sn_grid # class SanosPricer
        return r

from quantdata.widb.options_filter import MkArbFree, RawDayFilter, ExpiryFilter, OptionsFilter, filter_options_day

def assess_fit(d, ticker = "SPX", smoothness : float = 0.25, show_fit : bool = True, verbose: Context = Context.all ):
    d = to_date(d)
    raw_day_filter = RawDayFilter()
    expiry_filter = ExpiryFilter()
    mk_arb_free = MkArbFree()
    options_filter = OptionsFilter()

    data    = filter_options_day(ticker,d, mk_arb_free=mk_arb_free, options_filter=options_filter, raw_day_filter=raw_day_filter, expiry_filter=expiry_filter, verbose=verbose )

    strikes = data['pure_strike'].to_numpy()
    ixs     = np.argsort(strikes)
    strikes = strikes[ixs]
    tte_y   = data['tte_y'].to_numpy()[ixs]
    bid     = data['pure_bid_price'].to_numpy()[ixs]
    ask     = data['pure_ask_price'].to_numpy()[ixs]
    del ixs

    # split into per-expiry
    # ---------------------

    market = PrettyObject()
    market.uid      = unique_hash8( raw_day_filter, expiry_filter, options_filter, mk_arb_free )
    market.expiries = np.unique(tte_y)
    market.strikes  = []
    market.bids     = []
    market.asks     = []

    for expiry in market.expiries:
        mask = tte_y == expiry
        market.strikes.append( strikes[mask] )
        market.bids.append( bid[mask] )
        market.asks.append( ask[mask] )

    # NMS fit
    # -------
    sanos_bench = SanosBenchmarkConfig(smoothness=smoothness)
    r          = sanos_bench( ticker, d, market=market, message_context=lambda : f"{ticker} {d}", verbose=verbose, override_cache_mode="off" )

    str_smooth = f"{sanos_bench.smoothness:.2f}-smooth" if sanos_bench.smoothness is not None else "Linear"

    with figure(f"NMS {sanos_bench.grid_nms} {str_smooth} Fits {d} ", col_size=8, row_size=6, columns=4 ) as fig:
        X, Y = np.meshgrid(r.nms, r.expiries*DAYS_PER_YEAR)
        axVols = fig.add_subplot("Interpolated Market Vols", projection="3d")
        axVols.plot_surface(X, Y, r.mid_vols, cmap='viridis', edgecolor='none')
        axVols.set_ylabel("Days")
        axVols.set_xlabel("NMS")

        axVols = fig.add_subplot("Fitted Vols", projection="3d")
        axVols.plot_surface(X, Y, r.fit_vols, cmap='viridis', edgecolor='none')
        axVols.set_ylabel("Days")
        axVols.set_xlabel("NMS")

        axVolD = fig.add_subplot("|Fitted - Interpolated Market Vols|", projection="3d")
        axVolD.plot_surface(X, Y, np.abs( r.fit_vols - r.mid_vols ), cmap='viridis', edgecolor='none')
        axVolD.set_ylabel("Days")
        axVolD.set_xlabel("NMS")

        axVols = fig.add_subplot("DLVs", projection="3d")
        axVols.plot_surface(X, Y, np.sqrt(r.dlvsqs), cmap='viridis', edgecolor='none')
        axVols.set_ylabel("Days")
        axVols.set_xlabel("NMS")

        axVolsGr = fig.add_subplot("DLVs (strike)")
        x = []
        y = []
        z = []
        for t, expiry in enumerate(r.expiries):
            x.append(r.strikes[t,1:-1]*0.+expiry*DAYS_PER_YEAR) # (N-2,)
            y.append(r.strikes[t,1:-1])
            z.append(np.sqrt(r.dlvsqs[t,1:-1]))
        x = np.concatenate(x)
        y = np.concatenate(y)
        z = np.concatenate(z)
        p = axVolsGr.scatter(x=x, y=y, c=z, cmap="viridis", s=10 )
        fig.colorbar(p, ax=axVolsGr, label="DLVs")
        axVolsGr.set_xlabel("Days")
        axVolsGr.set_ylabel("Strike/Forward")
        #axVolsGr.set_xscale("log")

        axDensity = fig.add_subplot("Densities")
        x = []
        y = []
        z = []
        for t, expiry in enumerate(r.pricer.expiries):
            x.append(r.pricer.xstrikes[t][1:-1]*0.+expiry*DAYS_PER_YEAR)
            y.append(r.pricer.xstrikes[t][1:-1])
            z.append(r.pricer.xdensities[t][1:-1])
        x = np.concatenate(x)
        y = np.concatenate(y)
        z = np.concatenate(z)
        p = axDensity.scatter(x=x, y=y, c=z, cmap="viridis", s=10 )
        fig.colorbar(p, ax=axDensity, label="Densities")
        axDensity.set_xlabel("Days (log)")
        axDensity.set_ylabel("Strike/Forward")
        axDensity.set_xscale("log")

        axDensity = fig.add_subplot("Densities", projection="3d")
        x = []
        y = []
        z = []
        for t, expiry in enumerate(r.pricer.expiries):
            x.append(r.pricer.xstrikes[t][1:-1]*0.+expiry*DAYS_PER_YEAR)
            y.append(r.pricer.xstrikes[t][1:-1])
            z.append(r.pricer.xdensities[t][1:-1])
        x = np.concatenate(x)
        y = np.concatenate(y)
        z = np.concatenate(z)
        from matplotlib import cm
        from matplotlib.colors import Normalize
#        norm = Normalize(vmin=z.min()-0.0001, vmax=z.max())
#        colors = cm.viridis(norm(z))
#        mappable = cm.ScalarMappable(norm=norm, cmap="viridis")
#        mappable.set_array(z)
        z += 0.00001
        x  = np.log(x)
        dx = (x.max() - x.min())/100.
        dy = (y.max() - y.min())/100.
                         
        p = axDensity.bar3d(x,y,z*0., dx=dx, dy=dy, dz=z, cmap="viridis", edgecolor='none' )
#        fig.colorbar(mappable, ax=axDensity, shrink=0.6, label="f(x, y)")        
        axDensity.set_xlabel("log-Days")
        axDensity.set_ylabel("Strike/Forward")
        axDensity.set_zlabel("Density")

        axATM = fig.add_subplot("ATM Vols")
        axATM.plot( r.expiries*DAYS_PER_YEAR, r.atm_vols, "*-", label="ATM Vols")
        axATM.plot( r.expiries*DAYS_PER_YEAR, r.fwd_vols, "*-", label="Fwd Vols")
        axATM.legend(loc="lower right")
        axATM.set_ylabel("Vol")
        axATM.set_xlabel("Days")
        axATM.set_ylim(0., max(np.max(r.atm_vols), np.max(r.fwd_vols))*1.1 )
    


af = assess_fit("2023-02-13", smoothness=None)

#af = assess_fit("2025-07-01", smoothness=None)
#af = assess_fit("2025-07-01", smoothness=0.25)




#af = assess_fit("2024-12-26")



00: Collecting raw market data for SPX on 2023-02-13.


RuntimeError: Failed to connect to WRDS with username 'hansbuehler': (psycopg2.OperationalError) connection to server at "wrds-pgdata.wharton.upenn.edu" (165.123.60.118), port 9737 failed: FATAL:  PAM authentication failed for user "hansbuehler"

(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [ ]:
from quantdata.biz_calendar import DateRange, dt
dates = DateRange(start=dt.date(2020, 1, 1), end=dt.date(2025, 7, 1)).get_business_days()
dates = dates[::60]


for d in dates:
    print(f"\rAssessing fit for date {d}... ", end="")
    assess_fit(d,verbose=Context.quiet )